In [1]:
from datasets import load_dataset
import pandas as pd
import re
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

# =========================
# 1. LOAD DATASET
# =========================
dataset = load_dataset("stanfordnlp/imdb")
train_df = pd.DataFrame(dataset["train"])

print("\nShape:", train_df.shape)
print("\nLabel distribution:")
print(train_df["label"].value_counts())

# =========================
# 2. FEATURES (EDA)
# =========================
train_df["length"] = train_df["text"].apply(len)
train_df["word_count"] = train_df["text"].apply(lambda x: len(x.split()))

print("\nStats:")
print(train_df[["length", "word_count"]].describe())

# =========================
# 3. CLEANING TEXT
# =========================
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = " ".join([w for w in text.split() if w not in stop_words])
    text = re.sub(r"\s+", " ", text)
    return text.strip()

train_df["clean_text"] = train_df["text"].apply(clean_text)

# Après le nettoyage
X = train_df["clean_text"]
y = train_df["label"]

# =========================
# 4. VOCABULARY
# =========================
all_words = " ".join(train_df["clean_text"]).split()
counter = Counter(all_words)

print("\nTop 20 words:")
print(counter.most_common(20))

# =========================
# 5. PLOTS (SAVE ONLY)
# =========================

# Sentiment distribution
sns.countplot(x="label", data=train_df)
plt.title("Sentiment Distribution")
plt.savefig("sentiment_distribution.png")
plt.close()

# Word length distribution
plt.hist(train_df["word_count"], bins=50)
plt.title("Word Count Distribution")
plt.savefig("word_distribution.png")
plt.close()

# =========================
# 6. FINAL CHECK
# =========================
print("\nClean text sample:")
print(train_df["clean_text"].head())

# Sauvegarde du dataset nettoyé
train_df.to_csv("imdb_cleaned.csv", index=False)

print("Dataset sauvegardé : imdb_cleaned.csv")

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]


Shape: (25000, 2)

Label distribution:
label
0    12500
1    12500
Name: count, dtype: int64

Stats:
            length    word_count
count  25000.00000  25000.000000
mean    1325.06964    233.787200
std     1003.13367    173.733032
min       52.00000     10.000000
25%      702.00000    127.000000
50%      979.00000    174.000000
75%     1614.00000    284.000000
max    13704.00000   2470.000000

Top 20 words:
[('movie', 44030), ('film', 40147), ('one', 26788), ('like', 20274), ('good', 15140), ('time', 12723), ('even', 12646), ('would', 12436), ('story', 11983), ('really', 11736), ('see', 11475), ('well', 10662), ('much', 9765), ('get', 9310), ('bad', 9301), ('people', 9285), ('also', 9156), ('first', 9061), ('great', 9058), ('made', 8362)]

Clean text sample:
0    rented curious yellow video store controversy ...
1    curious yellow risible pretentious steaming pi...
2    avoid making type film future film interesting...
3    film probably inspired godard masculin f minin...
4    oh 

In [2]:
!pip install transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.8 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.

In [3]:
!pip install nltk

In [4]:
import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Charger les données
df = pd.read_csv("imdb_cleaned.csv")

# Séparation
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

# CHANGER ICI
model_name = "huawei-noah/TinyBERT_General_4L_312D"
# model_name = "google/mobilebert-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Conversion HF Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

def tokenize(batch):
    return tokenizer(
        batch["clean_text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

# =========================
# Sauvegarder le modèle
# =========================

trainer.save_model("./tinybert_model")
tokenizer.save_pretrained("./tinybert_model")

print("Modèle sauvegardé dans ./tinybert_model")

results = trainer.evaluate()

print(results)

results = trainer.evaluate()

print("\nRésultats finaux :")
print(results)

# Sauvegarder les résultats dans un fichier texte
with open("tinybert_results.txt", "w") as f:
    f.write(str(results))

print("Résultats sauvegardés dans tinybert_results.txt")

config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

...

Modèle sauvegardé dans ./tinybert_model


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.6258333325386047, 'eval_accuracy': 0.8724, 'eval_f1': 0.8711111111111111, 'eval_runtime': 6.8485, 'eval_samples_per_second': 730.089, 'eval_steps_per_second': 45.704, 'epoch': 1.0}


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Résultats finaux :
{'eval_loss': 0.6258333325386047, 'eval_accuracy': 0.8724, 'eval_f1': 0.8711111111111111, 'eval_runtime': 7.5033, 'eval_samples_per_second': 666.377, 'eval_steps_per_second': 41.715, 'epoch': 1.0}
Résultats sauvegardés dans tinybert_results.txt


In [5]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Charger les données
df = pd.read_csv("imdb_cleaned.csv")

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_dataset = Dataset.from_pandas(
    train_df[["clean_text", "label"]]
)
test_dataset = Dataset.from_pandas(
    test_df[["clean_text", "label"]]
)

# MobileBERT
model_name = "google/mobilebert-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["clean_text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

training_args = TrainingArguments(
    output_dir="./mobilebert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    learning_rate=2e-5,
    logging_steps=500
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

# =========================
# Sauvegarder le modèle
# =========================

trainer.save_model("./mobilebert_model")
tokenizer.save_pretrained("./mobilebert_model")

print("Modèle sauvegardé dans ./mobilebert_model")

results = trainer.evaluate()
print(results)

results = trainer.evaluate()

print("\nRésultats finaux :")
print(results)

# Sauvegarder les résultats dans un fichier texte
with open("mobilebert_results.txt", "w") as f:
    f.write(str(results))

print("Résultats sauvegardés dans mobilebert_results.txt")

config.json:   0%|          | 0.00/847 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

...

Modèle sauvegardé dans ./mobilebert_model


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.7813645601272583, 'eval_accuracy': 0.871, 'eval_f1': 0.8697232882246011, 'eval_runtime': 38.1968, 'eval_samples_per_second': 130.901, 'eval_steps_per_second': 8.194, 'epoch': 1.0}


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Résultats finaux :
{'eval_loss': 0.7813645601272583, 'eval_accuracy': 0.871, 'eval_f1': 0.8697232882246011, 'eval_runtime': 37.6515, 'eval_samples_per_second': 132.797, 'eval_steps_per_second': 8.313, 'epoch': 1.0}
Résultats sauvegardés dans mobilebert_results.txt


In [6]:
import pandas as pd
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }
    
df = pd.read_csv("imdb_cleaned.csv")

print(df.head())


train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)


tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

def tokenize(batch):
    return tokenizer(
        batch["clean_text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)
train_dataset = train_dataset.remove_columns(
    ["text", "clean_text", "length", "word_count"]
)

test_dataset = test_dataset.remove_columns(
    ["text", "clean_text", "length", "word_count"]
)

train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

train_dataset.set_format("torch")
test_dataset.set_format("torch")
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch"
)

from transformers import Trainer

trainer = Trainer(
     model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)
trainer.train()

results = trainer.evaluate()

print(results)

                                                text  label  length  \
0  I rented I AM CURIOUS-YELLOW from my video sto...      0    1640   
1  "I Am Curious: Yellow" is a risible and preten...      0    1294   
2  If only to avoid making this type of film in t...      0     528   
3  This film was probably inspired by Godard's Ma...      0     706   
4  Oh, brother...after hearing about this ridicul...      0    1814   

   word_count                                         clean_text  
0         288  rented curious yellow video store controversy ...  
1         214  curious yellow risible pretentious steaming pi...  
2          93  avoid making type film future film interesting...  
3         118  film probably inspired godard masculin f minin...  
4         311  oh brother hearing ridiculous film umpteen yea...  


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.601800,0.493568,0.901400,0.901301


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.4935675859451294, 'eval_accuracy': 0.9014, 'eval_f1': 0.9013013013013013, 'eval_runtime': 25.3456, 'eval_samples_per_second': 197.273, 'eval_steps_per_second': 12.349, 'epoch': 1.0}
